In [1]:
import ROOT
ROOT.EnableImplicitMT(16)
import pandas as pd
import libPy
weights = [
    "hw_nominal",     # nominal MC weight.                  scalar
    "hw_alphaS_up",   # up alpha_s variaton for PHD4LHC;    scalar
    "hw_alphaS_dn",   # down alpha_s variaton for PHD4LHC;  scalar
    "hw_pdf4lhc_unc", # 30 Eigen variation for PHD4LHC      vector
    "hw_qcd",         # muR/muF variation for the given MC; vector
]

In [ ]:
dfs = {}
dfs['ALL']      = ROOT.RDataFrame("tree", "ntuples/mc23_vbf_hyy_stxs.root")
dfs['UNKNOWN']  = dfs['ALL'].Filter("HTXS_Stage1_2_Fine_Category_pTjet30 == 0")

for val, category in libPy.stage_1_2_fine['vbf'].items():
    dfs[category] = dfs['ALL'].Filter(f"HTXS_Stage1_2_Fine_Category_pTjet30 == {val}")

# # determine the length of the vector weights, which should be the same for all events
# len_hw_pdf4lhc_unc = set(dfs['ALL'].Range(10).Define('len_hw_pdf4lhc_unc', 'hw_pdf4lhc_unc.size()').AsNumpy(['len_hw_pdf4lhc_unc'])['len_hw_pdf4lhc_unc'])
# len_hw_qcd         = set(dfs['ALL'].Range(10).Define('len_hw_qcd', 'hw_qcd.size()').AsNumpy(['len_hw_qcd'])['len_hw_qcd'])
# assert (len(len_hw_pdf4lhc_unc) == 1 and len(len_hw_qcd) == 1)
# len_hw_pdf4lhc_unc = list(len_hw_pdf4lhc_unc)[0]
# len_hw_qcd         = list(len_hw_qcd)[0]
len_hw_pdf4lhc_unc, len_hw_qcd = 41, 8

In [3]:
weight_dict = {}
futures = []
for slice, df in dfs.items():
    weight_dict[slice] = {}
    for weight in weights:
        if weight == "hw_pdf4lhc_unc":
            for i in range(len_hw_pdf4lhc_unc):
                weight_name = f"{weight}_{i}"
                weight_dict[slice][weight_name] = df.Define(weight_name, f"{weight}.at({i})").Filter(f"{weight_name} == {weight_name}").Sum(weight_name)
                futures.append(weight_dict[slice][weight_name])
        elif weight == "hw_qcd":
            for i in range(len_hw_qcd):
                weight_name = f"{weight}_{i}"
                weight_dict[slice][weight_name] = df.Define(weight_name, f"{weight}.at({i})").Filter(f"{weight_name} == {weight_name}").Sum(weight_name)
                futures.append(weight_dict[slice][weight_name])
        else:
            weight_dict[slice][weight] = df.Filter(f"{weight} == {weight}").Sum(weight)
            futures.append(weight_dict[slice][weight])
ROOT.RDF.RunGraphs(futures)

1

In [4]:
for slice, weight_sum_dict in weight_dict.items():
    for weight_name, weight_sum in weight_sum_dict.items():
        weight_dict[slice][weight_name] = weight_sum.GetValue()

In [5]:
pdf = pd.DataFrame(weight_dict)
pdf = pdf.apply(lambda row : (row / row['ALL']), axis=1)
ratio_pdf = pdf.apply(lambda row : row / pdf.iloc[0], axis=1)
# pdf.columns = [col + '_acc' for col in pdf.columns]
# pdf[[col.split('_')[0] + '_xs' for col in pdf.columns]] = pdf.apply(lambda row : row * xs, axis=1)
# pdf = pd.concat([pdf, ratio_pdf.add_suffix('_ratio')], axis=1)
pdf.to_csv("res_stxs/run3_vbf_stxs.csv", index=True)

In [7]:
import pandas as pd
pdf = pd.read_csv("res_stxs/run3_vbf_stxs.csv", index_col=0)

official = {key.upper() : val for key, val in libPy.official_1_2_fine_run2['vbf'].items()}
official['UNKNOWN'] = 100 - sum(official.values())
mine = pdf.iloc[0] * 100
mine.index = [index.upper() for index in mine.index]
comp_df = pd.DataFrame.from_dict(official, orient='index')
comp_df.columns = ['official_run2']
comp_df['mine'] = mine
comp_df['diff'] = comp_df['mine'] - comp_df['official_run2']
comp_df['diff_pct'] = abs(comp_df['diff'] / comp_df['official_run2'] * 100)
comp_df

,official_run2,mine,diff,diff_pct
QQ2HQQ_FWDH,7.035200,7.499444,0.464244,6.598873
QQ2HQQ_0J,7.392270,7.197982,-0.194288,2.628265
QQ2HQQ_1J,34.197800,33.823555,-0.374245,1.094355
QQ2HQQ_GE2J_MJJ_0_60_PTHJJ_0_25,0.505031,0.491757,-0.013274,2.628299
QQ2HQQ_GE2J_MJJ_60_120_PTHJJ_0_25,0.997949,0.992838,-0.005111,0.512135
QQ2HQQ_GE2J_MJJ_120_350_PTHJJ_0_25,7.549220,7.487977,-0.061243,0.811245
QQ2HQQ_GE2J_MJJ_0_60_PTHJJ_GT25,0.808783,0.779830,-0.028953,3.579868
QQ2HQQ_GE2J_MJJ_60_120_PTHJJ_GT25,1.285060,1.297829,0.012769,0.993643
QQ2HQQ_GE2J_MJJ_120_350_PTHJJ_GT25,3.642060,3.615327,-0.026733,0.734005
QQ2HQQ_GE2J_MJJ_350_700_PTH_0_200_PTHJJ_0_25,10.314500,10.291459,-0.023041,0.223386
